# Introduction to Object-Oriented Programming (OOP)

Welcome to the OOP section! In this notebook, we will cover the 5 fundamental pillars of Object-Oriented Programming.

To make things practical, we will build examples based on a **Fintech / Banking System**, completely avoiding the classic (but sometimes confusing) "Animal/Dog/Cat" examples.

## 1. Classes and Objects

- **Class**: A blueprint or template for creating objects. It defines the structure and behaviors (methods).
- **Object**: An instance of a class. It contains actual data.

Here, `BankAccount` is the blueprint, and `my_account` is the actual object.

In [ ]:
class BankAccount:
    # The __init__ method initializes the object's state
    def __init__(self, owner: str, initial_balance: float):
        self.owner = owner
        self.balance = initial_balance

    def deposit(self, amount: float):
        self.balance += amount
        return f"Deposited ${amount}. New balance: ${self.balance}"

# Creating an object (Instance)
my_account = BankAccount(owner="Alice", initial_balance=100.0)
print(my_account.deposit(50.0))

## 2. Encapsulation

Encapsulation is the practice of **hiding internal state** and requiring all interaction to be performed through an object's methods. 

In a bank, you shouldn't be able to manually edit your balance like `account.balance = 1000000`. You must go through the proper channels (`deposit` or `withdraw`). In Python, we indicate private attributes using a double underscore `__`.

In [ ]:
class SecureBankAccount:
    def __init__(self, owner: str, initial_balance: float):
        self.owner = owner
        self.__balance = initial_balance  # The double underscore makes it private

    # Getter method to safely read the balance
    def get_balance(self):
        return self.__balance

    # Method to safely modify the balance
    def withdraw(self, amount: float):
        if amount > self.__balance:
            return "Insufficient funds!"
        self.__balance -= amount
        return f"Withdrawal successful. Remaining balance: ${self.__balance}"

secure_account = SecureBankAccount("Bob", 500.0)

# print(secure_account.__balance)  # <--- This would throw an AttributeError!
print("Secure balance:", secure_account.get_balance())
print(secure_account.withdraw(1000.0))

## 3. Inheritance

Inheritance allows a new class to inherit attributes and methods from an existing class. This promotes **code reusability**.

For example, a `SavingsAccount` is exactly like a `SecureBankAccount`, but it adds a new feature: interest rates.

In [ ]:
class SavingsAccount(SecureBankAccount):  # Inherits from SecureBankAccount
    def __init__(self, owner: str, initial_balance: float, interest_rate: float):
        # Call the parent class's __init__ method
        super().__init__(owner, initial_balance)
        self.interest_rate = interest_rate

    def apply_interest(self):
        # We use the public getter because __balance is private in the parent class
        current_balance = self.get_balance()
        interest = current_balance * self.interest_rate
        
        # Wait, how do we add money if we don't have a deposit method in SecureBankAccount? 
        # In a real system, the parent class MUST provide a protected way to update the balance.
        # For this example, let's just print what the interest would be:
        return f"Interest applied: ${interest}"

savings = SavingsAccount("Charlie", 1000.0, 0.05)
print(savings.get_balance()) # Reusing the parent's method
print(savings.apply_interest()) # Using the child's exclusive method

## 4. Polymorphism

Polymorphism means "many forms". It allows different classes to have methods with the exact same name, but different behaviors. 

This means a system can interact with multiple different classes using the **same interface** (without needing to know the exact class type).

In [ ]:
class EmailNotification:
    def send(self, message: str):
        return f"Sending EMAIL: {message}"

class SMSNotification:
    def send(self, message: str):
        return f"Sending SMS: {message}"

class PushNotification:
    def send(self, message: str):
        return f"Sending PUSH NOTIFICATION: {message}"

# This function expects ANY object that has a .send() method.
# It doesn't care if it's an Email, SMS, or Push. That is polymorphism!
def notify_user(notifier, message):
    print(notifier.send(message))

# We can easily swap implementations without breaking our system
notify_user(EmailNotification(), "Your statement is ready")
notify_user(SMSNotification(), "Fraud alert: Did you spend $50?")

## 5. Abstraction

Abstraction is about hiding the complex background details and showing only the essential features to the user. It also forces subclasses to follow a strict blueprint.

In Python, we use the `abc` (Abstract Base Classes) module to enforce this. An abstract class cannot be instantiated directly; it serves purely as a strict template.

In [ ]:
from abc import ABC, abstractmethod

# This is a strict blueprint. You cannot create a "PaymentProcessor" directly.
class PaymentProcessor(ABC):
    
    @abstractmethod
    def process_payment(self, amount: float):
        pass

# If we don't implement process_payment in this class, Python will throw an error
class StripeProcessor(PaymentProcessor):
    def process_payment(self, amount: float):
        return f"Processing ${amount} through Stripe's secure API..."

class PayPalProcessor(PaymentProcessor):
    def process_payment(self, amount: float):
        return f"Redirecting to PayPal to process ${amount}..."

# processor = PaymentProcessor()  <--- This will throw a TypeError!

processor1 = StripeProcessor()
print(processor1.process_payment(25.99))

processor2 = PayPalProcessor()
print(processor2.process_payment(50.00))